# SCOPE: Semantic COdec Pre-Editing for Object-Preserving Compression (ACCV 2026)

Notebook chạy thực nghiệm trên **Google Colab / Colab+** ([Mở trực tiếp trên Google Colab](https://colab.research.google.com/github/munnn01/scope_again/blob/main/notebooks/SCOPE_Colab.ipynb)) cho kho mã nguồn đã đối chiếu thông số bài báo [`munnn01/scope_again`](https://github.com/munnn01/scope_again) (dựa trên bản chính thức [`miccunifi/SCOPE`](https://github.com/miccunifi/SCOPE), `arXiv:2610.39778`).

- Xem bảng đối chiếu chi tiết toàn bộ phương trình, kiến trúc, 11 checkpoint chính thức và siêu tham số tại `docs/PAPER_VERIFICATION.md`.
- Chọn Runtime **GPU** (`Runtime -> Change runtime type -> T4 / L4 / A100 GPU`) trước khi chạy.

In [ ]:
# 1. Kiểm tra GPU, clone repo (an toàn khi chạy lại cell) và cài đặt thư viện cho Google Colab / Colab+
!nvidia-smi
import os
if not os.path.isfile('requirements-colab.txt'):
    if os.path.isfile('../requirements-colab.txt'):
        %cd ..
    else:
        if not os.path.exists('scope_again'):
            !git clone https://github.com/munnn01/scope_again.git
        %cd scope_again

# Giữ nguyên PyTorch CUDA & NumPy 2.x có sẵn trên Colab+, cài đặt phụ thuộc SCOPE không xung đột ABI:
# (CompressAI 1.2.8 cài với --no-build-isolation --no-deps ngay sau khi đủ gói phụ trợ để không hạ cấp NumPy xuống 1.26.x)
!pip install -q -r requirements-colab.txt
!pip install -q --no-build-isolation --no-deps compressai==1.2.8
!apt-get update -qq && apt-get install -y -qq libopenjp2-tools

# Xác minh nhanh các thư viện cốt lõi, C++ extensions & ABI ngay trên phiên chạy hiện tại (không cần Restart Runtime):
!python -c "import torch, numpy, cv2, compressai, ultralytics, diff_jpeg, kornia, pycocotools; from compressai import _CXX, ans; print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| NumPy:', numpy.__version__, '| OpenCV:', cv2.__version__, '| CompressAI:', compressai.__version__)"

In [ ]:
# (Tùy chọn) Cài Detectron2 khi cần huấn luyện từ đầu với giáo viên Faster R-CNN X101-FPN
# hoặc chấm điểm lại ảnh nén bằng Faster R-CNN:
# !pip install --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'

# (Tùy chọn) Biên dịch thực thi BPG (bpgenc / bpgdec) trong /tmp trên Ubuntu 22.04/24.04 (GCC 11–14 & CMake 3.x/4.x)
# khi chạy đường cong BPG thực tế trong configs/eval.yaml:
# !apt-get update -qq && apt-get install -y -qq libpng-dev libjpeg-dev yasm cmake
# !rm -rf /tmp/libbpg_build && mkdir -p /tmp/libbpg_build && wget -q -O /tmp/libbpg_build/libbpg.tar.gz https://bellard.org/bpg/libbpg-0.9.8.tar.gz && tar -xzf /tmp/libbpg_build/libbpg.tar.gz -C /tmp/libbpg_build && sed -i 's/ OLD)/ NEW)/g' /tmp/libbpg_build/libbpg-0.9.8/x265/source/CMakeLists.txt && sed -i 's/\bregister //g' /tmp/libbpg_build/libbpg-0.9.8/x265/source/common/md5.cpp && CMAKE_POLICY_VERSION_MINIMUM=3.5 make -C /tmp/libbpg_build/libbpg-0.9.8 -j$(nproc) CMAKE_OPTS="-DCMAKE_CXX_STANDARD=14" USE_BPGVIEW= install && rm -rf /tmp/libbpg_build && command -v bpgenc bpgdec

In [ ]:
# 2. Chạy bộ kiểm thử tự động & kiểm tra cấu hình thông số bài báo
!python tools/run_full_tests.py
!python tools/validate_environment.py --mode static --show-ok

In [ ]:
# 3. Tải trọng số SCOPE chính thức từ Google Drive của tác giả (hỗ trợ đủ 11/11 mô hình bài báo)
import os
import gdown

OFFICIAL_GDRIVE_CHECKPOINTS = {
    'FULL': ('1M7lHWbNjrckMNBC623Cn5PvBDuDUBlda', 'runs/preedit_d2fpn_nojp2_unbiased_linearsev_mediumbpg_sevbasis_scratch_FULL_20260701_140557/best_val.pt'),
    'NOBG': ('1h7_LilISRo1YkVC9_W_-XsSTIgkl-UFt', 'runs/preedit_d2fpn_nojp2_unbiased_linearsev_mediumbpg_sevbasis_scratch_NOBG_20260702_220324/best_val.pt'),
    'NODET': ('16gWR89Z6bT1ye4lAaXUj9El6J_PwN9my', 'runs/preedit_d2fpn_nojp2_unbiased_linearsev_mediumbpg_sevbasis_scratch_NODET_20260702_164808/best_val.pt'),
    'FAMILYONLY': ('1aMxhoUO4Zh6yR90bCLPN8T3p17Ubwmhs', 'runs/preedit_d2fpn_nojp2_unbiased_linearsev_mediumbpg_familyonly_scratch_20260828_165524/best_val.pt'),
    'SEVERITYONLY': ('1k3HSKkNwYMcx60q6tqun02gHwBNKCJNQ', 'runs/preedit_d2fpn_nojp2_unbiased_linearsev_mediumbpg_SEVERITYONLY_scratch_20260829_022023/best_val.pt'),
    'JPEG_UNCOND': ('1l0l9pqqmoZNtNCG59f9s_r0cYkE3Vg7l', 'runs/preedit_d2fpn_JPEG_UNCONDITIONED_scratch_20260829_140650/best_val.pt'),
    'BPG_UNCOND': ('1SEWVGE2-PdrrIsF9jrmCoi4rCfsnoh4z', 'runs/preedit_d2fpn_BPG_UNCONDITIONED_scratch_20260829_184336/best_val.pt'),
    'BMSHJ_UNCOND': ('1gWES4qI_6W8N7my0KdML5EuBdKcb4JbG', 'runs/preedit_d2fpn_BMSHJ_UNCONDITIONED_scratch_20260829_224613/best_val.pt'),
    'JPEG_SEV': ('1mZWYleovrVdp9aW23I_bWnBHehf-BT6R', 'runs/preedit_d2fpn_JPEG_SEVERITYONLY_percodec_scratch_20260830_131351/best_val.pt'),
    'BPG_SEV': ('1SD_GAEi_F2ERUT75MFYsfxnM2IlNVf5H', 'runs/preedit_d2fpn_BPG_SEVERITYONLY_percodec_scratch_20260830_180851/best_val.pt'),
    'BMSHJ_SEV': ('1_4S0zMQHUiNrhK-CEDcuNMCZAd2qMOBD', 'runs/preedit_d2fpn_BMSHJ_SEVERITYONLY_percodec_scratch_20260830_223001/best_val.pt'),
}

file_id, ckpt_path = OFFICIAL_GDRIVE_CHECKPOINTS['FULL']
os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)
if not os.path.exists(ckpt_path):
    out = gdown.download(id=file_id, output=ckpt_path, quiet=False)
    assert out and os.path.isfile(ckpt_path), f'Failed to download {ckpt_path}'

!python tools/check_runtime.py --checkpoints

In [ ]:
# 4. Demo nhanh tiền xử lý SCOPE + nén JPEG thực tế trên 1 ảnh mẫu COCO
import urllib.request
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from scope.inference import load_scope_model
from scope.evaluation.utils import jpeg_encode_decode

sample_url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
urllib.request.urlretrieve(sample_url, 'sample_coco.jpg')
img_bgr = cv2.imread('sample_coco.jpg')
h, w = img_bgr.shape[:2]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt_path = globals().get('ckpt_path', 'runs/preedit_d2fpn_nojp2_unbiased_linearsev_mediumbpg_sevbasis_scratch_FULL_20260701_140557/best_val.pt')
model, fam_map = load_scope_model(ckpt_path, device)

jpeg_q = 20
severity = (99.0 - jpeg_q) / 98.0  # Chuẩn hóa tuyến tính đúng bài báo: q in [1, 99]
x = torch.from_numpy(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0
with torch.no_grad():
    edited, delta_obj, delta_bg, attn, _ = model(
        x,
        severity=torch.tensor([severity], device=device),
        codec_family=torch.tensor([fam_map['diff_jpeg']], device=device),
    )

edited_bgr = cv2.cvtColor((edited[0].clamp(0, 1).permute(1, 2, 0).cpu().numpy() * 255).round().astype(np.uint8), cv2.COLOR_RGB2BGR)
base_rec, base_bytes = jpeg_encode_decode(img_bgr, quality=jpeg_q)
scope_rec, scope_bytes = jpeg_encode_decode(edited_bgr, quality=jpeg_q)

base_bpp = base_bytes * 8.0 / (h * w)
scope_bpp = scope_bytes * 8.0 / (h * w)
print(f'JPEG q={jpeg_q} | Direct: {base_bpp:.4f} bpp ({base_bytes} B) | SCOPE: {scope_bpp:.4f} bpp ({scope_bytes} B) | Rate delta: {(scope_bpp/base_bpp - 1)*100:+.2f}%')

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)); axes[0].set_title('Original')
axes[1].imshow(attn[0, 0].cpu().numpy(), cmap='inferno', vmin=0, vmax=1); axes[1].set_title(f'Attention A (s={severity:.2f})')
axes[2].imshow(cv2.cvtColor(base_rec, cv2.COLOR_BGR2RGB)); axes[2].set_title(f'Direct JPEG q={jpeg_q} ({base_bpp:.3f} bpp)')
axes[3].imshow(cv2.cvtColor(scope_rec, cv2.COLOR_BGR2RGB)); axes[3].set_title(f'SCOPE + JPEG q={jpeg_q} ({scope_bpp:.3f} bpp)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 5. (Tùy chọn) Tải COCO 2017 val2017 + YOLO26x/RT-DETR-l (an toàn khi chạy lại cell) để đánh giá đường cong Rate-mAP
# !mkdir -p datasets/coco-2017/validation models
# !test -d datasets/coco-2017/validation/data || (wget -q -c http://images.cocodataset.org/zips/val2017.zip && unzip -q -o val2017.zip && mv val2017 datasets/coco-2017/validation/data)
# !test -f datasets/coco-2017/validation/labels.json || (wget -q -c http://images.cocodataset.org/annotations/annotations_trainval2017.zip && unzip -q -o annotations_trainval2017.zip annotations/instances_val2017.json && mv annotations/instances_val2017.json datasets/coco-2017/validation/labels.json)
# !python -c "import os, shutil; from ultralytics import YOLO, RTDETR; os.makedirs('models', exist_ok=True); [None if os.path.isfile(f'models/{w}') else (cls(w), shutil.move(w, f'models/{w}')) for cls, w in ((YOLO, 'yolo26x.pt'), (RTDETR, 'rtdetr-l.pt'))]"

# Chạy thử nghiệm nhanh trên tập con 20 ảnh cho JPEG & BMSHJ:
# !python tools/smoke_real.py --config configs/eval.yaml --images 20 --experiments jpeg_q10 scope_jpeg_q10 bmshj_q1 scope_bmshj_q1

In [ ]:
# 6. (Tùy chọn) Huấn luyện SCOPE từ đầu theo đúng thông số bài báo (cần COCO train2017 + Detectron2)
# !bash training/train_scope.sh